In [1]:
import pandas as pd
import requests

from pathlib import Path

In [7]:
url = "https://raw.githubusercontent.com/anilbhaila/llm-zoomcamp-finalproject/refs/heads/main/data/Ecommerce_FAQ_Chatbot_dataset.json"


In [8]:
def load_data(*args, **kwargs):
    """
    Extract data from URL. 
    
    """
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes
        
        json_data = response.json()

        faqs = json_data.get("questions")
        # Create a DataFrame
        df = pd.DataFrame(list(faqs))

        return df
    except Exception as e:
        print(f"An error occurred while reading the CSV file: {e}")
        return None

In [9]:
df = load_data()
df.head()

,question,answer
0,How can I create an account?,"To create an account, click on the 'Sign Up' b..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and..."
2,How can I track my order?,You can track your order by logging into your ...
3,What is your return policy?,Our return policy allows you to return product...
4,Can I cancel my order?,You can cancel your order if it has not been s...


In [10]:
import re


In [11]:
def transformToAddChunk(data: pd.DataFrame, *args, **kwargs):
    """
    Template code for a transformer block to add Chunk.

    """
    # Specify your transformation logic here

    rowNumber = 0
    documents = []

    for _, row in data.iterrows():
        number = str(rowNumber)
        rowNumber+=1
        question = str(row['question'])
        answer = str(row['answer'])

        sanitized_question = re.sub(r'\W', '_', question[:30]).lower()
        document_id = f"doc_{number}_{sanitized_question}"

        # Format the document string
        chunk = '\n'.join([
            f'question:\n{question}\n',
            f'answer:\n{answer}\n',
        ])

        documents.append({
            'chunk': chunk,
            'data': {
                'number': number,
                'question': question,
                'answer': answer
            },
            'document_id': document_id,
        })

    print(f'Documents: {len(documents)}')

    return documents

In [12]:
chunk_documents = transformToAddChunk(df)

Documents: 79


In [13]:
chunk_documents[0]

{'chunk': "question:\nHow can I create an account?\n\nanswer:\nTo create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.\n",
 'data': {'number': '0',
  'question': 'How can I create an account?',
  'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process."},
 'document_id': 'doc_0_how_can_i_create_an_account_'}

In [14]:
from typing import Dict, List
import spacy

def transformToAddTokensBySpacyNLP(documents: List[Dict], *args, **kwargs):
    """
    Template code for a transformer block to Lemmatize.
    """
    count = len(documents)
    print('Documents', count)

    nlp = spacy.load('en_core_web_sm')
    
    data = []

    for idx, document in enumerate(documents):
        document_id = document['document_id']
        #if idx % 100 == 0:
            #print(f'{idx + 1}/{count}')

        # Process the text chunk using spacy
        chunk = document['chunk']
        doc = nlp(chunk)
        tokens = [token.lemma_ for token in doc]

        data.append(
            dict(
                chunk=chunk,
                document_id=document_id,
                tokens=tokens,
                question=document['data']['question'],
                answer=document['data']['answer'],
            )
        )

    print('\nReturned Data', len(data))

    return data

In [15]:
lemmatize_documents = transformToAddTokensBySpacyNLP(chunk_documents)
#lemmatize_documents[0]

Documents 79

Returned Data 79


In [16]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
from typing import Dict, List

import numpy as np
import spacy

def transformToAddEmbeddingBySpacy(documents: List[Dict], *args, **kwargs) ->List[Dict]:
    """
    Template code for a transformer block to create embeddings.
    """
    # Specify your transformation logic here
    count = len(documents)
    print('Embedding Documents By Spacy', count)

    data = []

    for idx, document in enumerate(documents):
        document_id = document['document_id']
        #if idx % 100 == 0:
            #print(f'{idx + 1}/{count}')
        nlp = spacy.load('en_core_web_sm')
        tokens = document['tokens']
    
        # Combine tokens back into a single string of text used for embedding
        text = ' '.join(tokens)
        doc = nlp(text)
    
        # Average the word vectors in the doc to get a general embedding
        embedding = np.mean([token.vector for token in doc], axis=0).tolist()
    
        data.append(dict(
            chunk=document['chunk'],
            document_id=document['document_id'],
            question=document['question'],
            answer=document['answer'],
            embedding=embedding,
        ))

    return data

In [18]:
embedding_documentsBySpacy = transformToAddEmbeddingBySpacy(lemmatize_documents)
print(f'Spacy embedding dimension: {len(embedding_documentsBySpacy[0]["embedding"])}')

Embedding Documents By Spacy 79
Spacy embedding dimension: 96


In [19]:
from typing import Dict, List

import numpy as np
import spacy

def transformToAddEmbeddingByST(documents: List[Dict], *args, **kwargs) ->List[Dict]:
    """
    Template code for a transformer block to create embeddings by Sentence Transformer.
    """
    # Specify your transformation logic here
    count = len(documents)
    print('Embedding Documents By ST count:', count)

    data = []

    for idx, document in enumerate(documents):
        embedding = model.encode(document['chunk'])
    
        data.append(dict(
            chunk=document['chunk'],
            document_id=document['document_id'],
            question=document['question'],
            answer=document['answer'],
            embedding=embedding,
        ))

    return data

In [20]:
embedding_documentsByST = transformToAddEmbeddingByST(lemmatize_documents)
print(f'ST embedding dimension: {len(embedding_documentsByST[0]["embedding"])}')

Embedding Documents By ST count: 79
ST embedding dimension: 384


In [21]:
import json

from typing import Dict, List, Union

import numpy as np
from elasticsearch import Elasticsearch, helpers

def export_dataToIndex(documents: List[Dict[str, Union[Dict, List[int], str]]], *args, **kwargs):
    """
    Exports data to some source.
    
    """
    # Specify your data exporting logic here
    connection_string = kwargs.get('connection_string', 'http://localhost:9200')
    index_name = kwargs.get('index_name', 'documents')
    number_of_shards = kwargs.get('number_of_shards', 1)
    number_of_replicas = kwargs.get('number_of_replicas', 0)
    dimensions = kwargs.get('dimensions')

    if dimensions is None and len(documents) > 0:
        document = documents[0]
        dimensions = len(document.get('embedding'))
        print(f"Dimensions:{dimensions}")

    es_client = Elasticsearch(connection_string, request_timeout=60.0)

    print(f'Connecting to Elasticsearch at {connection_string}')

    index_settings = {
            "settings": {
                "number_of_shards": number_of_shards,
                "number_of_replicas": number_of_replicas,
            },
            "mappings": {
                "properties": {
                    "chunk": {"type": "text"},
                    "document_id": {"type": "text"},
                    "question": {"type": "text"},
                    "answer": {"type": "text"},
                    "embedding": {
                        "type": "dense_vector", 
                        "dims": dimensions,
                        "index": True,
                        "similarity": "cosine"
                    },
                }
            }
        }
    
    if es_client.indices.exists(index=index_name):
        es_client.indices.delete(index=index_name)
        print(f'Index {index_name} deleted')

    es_client.indices.create(index=index_name, body=index_settings)
    print('Index created with properties:')
    print(json.dumps(index_settings, indent=2))
    print('Embedding dimensions:', dimensions)

    count = len(documents)
    print(f'Preparing bulk indexing payload for {count} documents...')
    
    # 1. Build actions generator/list for the bulk helper
    bulk_actions = []
    for document in documents:
        # Prevent NumPy array conversion crashes
        if isinstance(document['embedding'], np.ndarray):
            document['embedding'] = document['embedding'].tolist()
            
        action = {
            "_index": index_name,
            "_source": document
        }
        bulk_actions.append(action)
        
    # 2. Execute bulk insertion in a single optimized pipeline operation
    print("Sending batch payload to Elasticsearch...")
    success, errors = helpers.bulk(es_client, bulk_actions)
    print(f"Successfully indexed {success} documents.")
    
    if errors:
        print(f"Warning: Encounted errors during indexing: {errors}")
        
    # 3. Force index refresh so elements are immediately available for search
    es_client.indices.refresh(index=index_name)

    return [d['embedding'] for d in documents[:1]]

In [22]:
!curl -s http://localhost:9200/_cluster/health?pretty

{
  "cluster_name" : "docker-cluster",
  "status" : "green",
  "timed_out" : false,
  "number_of_nodes" : 1,
  "number_of_data_nodes" : 1,
  "active_primary_shards" : 0,
  "active_shards" : 0,
  "relocating_shards" : 0,
  "initializing_shards" : 0,
  "unassigned_shards" : 0,
  "unassigned_primary_shards" : 0,
  "delayed_unassigned_shards" : 0,
  "number_of_pending_tasks" : 0,
  "number_of_in_flight_fetch" : 0,
  "task_max_waiting_in_queue_millis" : 0,
  "active_shards_percent_as_number" : 100.0
}


In [4]:
!curl -s "http://localhost:9200/_cluster/allocation/explain?pretty"

{
  "error" : {
    "root_cause" : [
      {
        "type" : "illegal_argument_exception",
        "reason" : "No shard was specified in the request which means the response should explain a randomly-chosen unassigned shard, but there are no unassigned shards in this cluster. To explain the allocation of an assigned shard you must specify the target shard in the request. See https://www.elastic.co/guide/en/elasticsearch/reference/8.17/cluster-allocation-explain.html for more information."
      }
    ],
    "type" : "illegal_argument_exception",
    "reason" : "No shard was specified in the request which means the response should explain a randomly-chosen unassigned shard, but there are no unassigned shards in this cluster. To explain the allocation of an assigned shard you must specify the target shard in the request. See https://www.elastic.co/guide/en/elasticsearch/reference/8.17/cluster-allocation-explain.html for more information."
  },
  "status" : 400
}


In [ ]:
!curl -X PUT "http://localhost:9200/_cluster/settings" -H 'Content-Type: application/json' -d'
{
  "persistent": {
    "cluster.routing.allocation.disk.watermark.low": "95%",
    "cluster.routing.allocation.disk.watermark.high": "98%",
    "cluster.routing.allocation.disk.watermark.flood_stage": "99%"
  }
}'

SyntaxError: f-string: expecting '}', or format specs (4123770517.py, line 5)

In [23]:
stEmbedding_indexed = export_dataToIndex(embedding_documentsByST,index_name="documents_st")
stEmbedding_indexed

Dimensions:384
Connecting to Elasticsearch at http://localhost:9200
Index created with properties:
{
  "settings": {
    "number_of_shards": 1,
    "number_of_replicas": 0
  },
  "mappings": {
    "properties": {
      "chunk": {
        "type": "text"
      },
      "document_id": {
        "type": "text"
      },
      "question": {
        "type": "text"
      },
      "answer": {
        "type": "text"
      },
      "embedding": {
        "type": "dense_vector",
        "dims": 384,
        "index": true,
        "similarity": "cosine"
      }
    }
  }
}
Embedding dimensions: 384
Preparing bulk indexing payload for 79 documents...
Sending batch payload to Elasticsearch...
Successfully indexed 79 documents.


[[-0.00113416719250381,
  -0.12893573939800262,
  -0.02183106169104576,
  0.025953097268939018,
  -0.04402286931872368,
  0.018380776047706604,
  0.007050221785902977,
  0.02937832660973072,
  0.018643056973814964,
  0.0022506772074848413,
  -0.03759142383933067,
  -0.06683802604675293,
  0.06715784966945648,
  -0.007149727549403906,
  0.030264394357800484,
  -0.030028818175196648,
  -0.10149011760950089,
  -0.02623187005519867,
  0.009549454785883427,
  0.029605425894260406,
  0.05041772872209549,
  -0.0965995043516159,
  -0.05003385245800018,
  0.009134571999311447,
  -0.022396691143512726,
  -0.0652521401643753,
  0.05307162553071976,
  0.06391444802284241,
  0.024932313710451126,
  0.07959100604057312,
  0.11508239805698395,
  -0.07984435558319092,
  0.050062939524650574,
  -0.034444842487573624,
  -0.008694835938513279,
  -0.0203609187155962,
  -0.08820894360542297,
  -0.00048545654863119125,
  -0.036247141659259796,
  0.00011376035399734974,
  -0.0798783153295517,
  -0.0769598633

In [24]:
specyEmbedding_indexed = export_dataToIndex(embedding_documentsBySpacy,index_name="documents_spacy")
specyEmbedding_indexed

Dimensions:96
Connecting to Elasticsearch at http://localhost:9200
Index created with properties:
{
  "settings": {
    "number_of_shards": 1,
    "number_of_replicas": 0
  },
  "mappings": {
    "properties": {
      "chunk": {
        "type": "text"
      },
      "document_id": {
        "type": "text"
      },
      "question": {
        "type": "text"
      },
      "answer": {
        "type": "text"
      },
      "embedding": {
        "type": "dense_vector",
        "dims": 96,
        "index": true,
        "similarity": "cosine"
      }
    }
  }
}
Embedding dimensions: 96
Preparing bulk indexing payload for 79 documents...
Sending batch payload to Elasticsearch...
Successfully indexed 79 documents.


[[-0.21343110501766205,
  -0.5658838748931885,
  0.08929450809955597,
  -0.02144671604037285,
  -0.11743323504924774,
  0.044704996049404144,
  0.2153482884168625,
  0.03439968079328537,
  -0.037103597074747086,
  0.1286245882511139,
  -0.04293552786111832,
  0.14869725704193115,
  0.13001641631126404,
  0.21767206490039825,
  0.5346853137016296,
  0.08476880937814713,
  -0.15936948359012604,
  -0.1102224662899971,
  0.2835747301578522,
  0.07442142069339752,
  0.025112507864832878,
  0.4940701723098755,
  0.11377029120922089,
  -0.09249747544527054,
  0.3497903645038605,
  -0.07383579015731812,
  0.2835350036621094,
  0.011244019493460655,
  0.09735123813152313,
  0.20336204767227173,
  -0.009408357553184032,
  0.14112085103988647,
  0.29277804493904114,
  -0.21818043291568756,
  0.22006219625473022,
  -0.03635493665933609,
  0.026387162506580353,
  -0.07717084139585495,
  -0.033911097794771194,
  -0.10444217920303345,
  -0.12649232149124146,
  0.26982399821281433,
  0.256779670715332

In [25]:
es_client = Elasticsearch('http://localhost:9200') 

index_name='documents_st'
try:
    result = es_client.count(index=index_name)
    print(f"ES Checking = Document count in {index_name}: {result['count']}")
except Exception as e:
    print(f"ES Checking = Error: {str(e)}")

ES Checking = Document count in documents_st: 79


In [26]:
es_client = Elasticsearch('http://localhost:9200') 

index_name='documents_spacy'
try:
    result = es_client.count(index=index_name)
    print(f"ES Checking = Document count in {index_name}: {result['count']}")
except Exception as e:
    print(f"ES Checking = Error: {str(e)}")

ES Checking = Document count in documents_spacy: 79


In [89]:
def get_vector(query):
    doc = nlp(query)
    tokens = [token.lemma_ for token in doc]
    text = ' '.join(tokens)
    doc_lemmatized = nlp(text)
    vector = np.mean([token.vector for token in doc_lemmatized], axis=0).tolist()
    return vector

In [90]:
query = "How can i create my account?"
query

'How can i create my account?'

In [91]:
nlp = spacy.load('en_core_web_sm')

vector = get_vector(query)
len(vector)

96

In [ ]:
search_query_knn = {
    "knn": {
        "field": "embedding",
        "query_vector": vector,
        "k": 5,
        "num_candidates": 10000,
    },
    "size": 5,
    "_source": ['document_id', 'question', 'answer'],
}

index_name="documents_spacy"
es_results = es_client.search(index=index_name, body=search_query_knn)

[elem['_score'] for elem in es_results["hits"]["hits"]]

[0.7769791, 0.7687049, 0.75467646, 0.7458626, 0.7372478]

In [ ]:
# Vector Search in documents_spacy index.
[hit["_source"] for hit in es_results["hits"]["hits"]]

[{'document_id': 'doc_4_can_i_cancel_my_order_',
  'question': 'Can I cancel my order?',
  'answer': 'You can cancel your order if it has not been shipped yet. Please contact our customer support team with your order details, and we will assist you with the cancellation process.'},
 {'document_id': 'doc_2_how_can_i_track_my_order_',
  'question': 'How can I track my order?',
  'answer': "You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment."},
 {'document_id': 'doc_8_can_i_change_my_shipping_addre',
  'question': 'Can I change my shipping address after placing an order?',
  'answer': 'If you need to change your shipping address, please contact our customer support team as soon as possible. We will do our best to update the address if the order has not been shipped yet.'},
 {'document_id': 'doc_15_do_you_have_a_loyalty_program_',
  'question': 'Do you have a loyalty program?',

In [98]:
vector_st = model.encode(query)

search_query_knn = {
    "knn": {
        "field": "embedding",
        "query_vector": vector_st,
        "k": 5,
        "num_candidates": 10000,
    },
    "size": 5,
    "_source": ['document_id', 'question', 'answer'],
}

index_name="documents_st"
es_results = es_client.search(index=index_name, body=search_query_knn)

[elem['_score'] for elem in es_results["hits"]["hits"]]

[0.90626216, 0.706826, 0.6207144, 0.56382585, 0.5634005]

In [99]:
# Vector Search in documents_st index.
[hit["_source"] for hit in es_results["hits"]["hits"]]

[{'document_id': 'doc_0_how_can_i_create_an_account_',
  'question': 'How can I create an account?',
  'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process."},
 {'document_id': 'doc_16_can_i_order_without_creating_a',
  'question': 'Can I order without creating an account?',
  'answer': 'Yes, you can place an order as a guest without creating an account. However, creating an account offers benefits such as order tracking and easier future purchases.'},
 {'document_id': 'doc_15_do_you_have_a_loyalty_program_',
  'question': 'Do you have a loyalty program?',
  'answer': 'Yes, we have a loyalty program where you can earn points for every purchase. These points can be redeemed for discounts on future orders. Please visit our website to learn more and join the program.'},
 {'document_id': 'doc_28_what_should_i_do_if_my_discoun',
  'question': 'What should I do if my discount cod

In [100]:
if isinstance(model, SentenceTransformer):
    print("model is an instance of SentenceTransformer")

model is an instance of SentenceTransformer
